In [1]:
# =============================================================================
# NOTEBOOK 4 — Serve Finetuned Model via ngrok
# HawkWatch Gemma Edition | Kaggle Gemma 4 Good Hackathon
#
# PURPOSE:
#   Load finetuned Gemma 4 model from HuggingFace
#   Expose it as a REST API via FastAPI + ngrok
#   Your local app calls this URL instead of Google AI Studio
#
# THIS IS THE FINAL STEP THAT CONNECTS:
#   Your local FastAPI app  →  this Kaggle notebook  →  finetuned Gemma 4
#
# KEEP THIS NOTEBOOK RUNNING DURING DEMO
# GPU: T4 x1 | Internet: ON
# KAGGLE ACCOUNT: Account 1 (save for demo day)
#
# AFTER THIS NOTEBOOK:
#   Update your local .env:
#     GEMMA_ENDPOINT=https://xxxx.ngrok-free.app
#   Everything else in your app stays the same
# =============================================================================
 
 
# -----------------------------------------------------------------------------
# CELL 1 — Install dependencies
# -----------------------------------------------------------------------------

!pip install unsloth --quiet
!pip install --no-deps transformers==5.5.0 --quiet
!pip install --no-deps --upgrade timm --quiet
!pip install fastapi uvicorn pyngrok nest-asyncio --quiet
!pip install "huggingface-hub>=1.5.0,<2.0" --upgrade --quiet
 
import torch
torch._dynamo.config.recompile_limit = 64
 
print("Done — RESTART KERNEL then run from Cell 2")
 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 29.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 102.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 11.0 MB/s eta 0:00:00
   ━

In [ ]:
# -----------------------------------------------------------------------------
# CELL 2 — Load finetuned model from HuggingFace
# -----------------------------------------------------------------------------
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch

# Must be set after every kernel restart — Unsloth's compiled cache exceeds
# dynamo's default recompile limit (8), causing FX tracing errors on inference.
torch._dynamo.config.recompile_limit = 64
 
HF_MODEL_ID = "Dharya/secure_sight-gemma4-security"  # ← update this
HF_TOKEN    = ""                        # ← update this
 
print(f"Loading finetuned model: {HF_MODEL_ID}")
print("Takes ~5-10 min first run...\n")
 
model, tokenizer = FastModel.from_pretrained(
    model_name      = HF_MODEL_ID,
    dtype           = None,
    max_seq_length  = 2048,
    load_in_4bit    = True,
    full_finetuning = False,
    device_map      = {"": torch.cuda.current_device()},
    token           = HF_TOKEN,
)
 
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
FastModel.for_inference(model)
 
print("Model loaded and ready for inference!")
gpu = torch.cuda.get_device_properties(0)
mem = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"GPU: {gpu.name} | Memory used: {mem} GB")
 

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading finetuned model: Dharya/secure_sight-gemma4-security
Takes ~5-10 min first run...

==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

Model loaded and ready for inference!
GPU: Tesla T4 | Memory used: 10.39 GB


In [3]:
# -----------------------------------------------------------------------------
# CELL 3 — Define inference function
# -----------------------------------------------------------------------------
import json
import re
import torch
 
FIELD_REMINDER = """
Return ONLY a JSON object with these exact fields:
{
  "scene_description": "detailed description of the scene",
  "activity_detected": "specific activity occurring",
  "persons_count": 0,
  "severity": "CRITICAL or WARNING or CLEAR",
  "category": "Crime or Medical Emergency or Suspicious Activity or Disaster or Normal",
  "confidence": 0-100,
  "recommended_action": "specific action for security personnel",
  "objects_of_interest": ["list", "of", "objects"]
}"""
 
INSTRUCTION = "You are a security monitoring AI. Analyze this surveillance scene description and generate a structured incident report as JSON."
 
 
def run_inference(scene_description: str, max_new_tokens: int = 400) -> dict:
    """
    Run finetuned Gemma 4 on a scene description.
    Returns parsed JSON dict or error dict.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{INSTRUCTION}{FIELD_REMINDER}\n\nScene: {scene_description}"
                }
            ]
        }
    ]
 
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_dict           = True,
        return_tensors        = "pt",
    ).to("cuda")
 
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            use_cache      = True,
            temperature    = 1.0,
            top_p          = 0.95,
            top_k          = 64,
        )
 
    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens = True,
    ).strip()
 
    # Parse JSON from response
    try:
        return json.loads(decoded)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', decoded, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
    return {"error": "JSON parse failed", "raw_response": decoded}
 
 
# Quick test
print("Testing inference function...")
test_result = run_inference(
    "Two men are arguing aggressively near a parked car. One is pointing at the other."
)
print(f"Severity: {test_result.get('severity')}")
print(f"Category: {test_result.get('category')}")
print(f"Action:   {test_result.get('recommended_action')}")
print("\nInference function working correctly")
 
 

Testing inference function...
Severity: CRITICAL
Category: Crime
Action:   Dispatch security personnel immediately to the location to de-escalate the situation and prevent physical violence.

Inference function working correctly


In [4]:
# -----------------------------------------------------------------------------
# CELL 4 — Build FastAPI server
# -----------------------------------------------------------------------------
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import time
 
app = FastAPI(
    title    = "HawkWatch Gemma 4 Inference API",
    version  = "1.0.0",
    docs_url = "/docs",
)
 
# Allow all origins — needed for local app to call this endpoint
app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["*"],
    allow_methods     = ["*"],
    allow_headers     = ["*"],
)
 
 
class AnalyzeRequest(BaseModel):
    scene_description: str
    max_new_tokens:    Optional[int] = 400
 
 
class HealthResponse(BaseModel):
    status:    str
    model:     str
    gpu:       str
    timestamp: float
 
 
@app.get("/")
def root():
    return {
        "service": "HawkWatch Gemma 4 API",
        "status":  "running",
        "endpoints": {
            "POST /analyze":  "Analyze scene description, get incident report",
            "GET  /health":   "Check server health",
            "GET  /docs":     "Interactive API documentation",
        }
    }
 
 
@app.get("/health", response_model=HealthResponse)
def health():
    return {
        "status":    "healthy",
        "model":     HF_MODEL_ID,
        "gpu":       torch.cuda.get_device_name(0),
        "timestamp": time.time(),
    }
 
 
@app.post("/analyze")
def analyze(request: AnalyzeRequest):
    """
    Analyze a surveillance scene description and return a structured incident report.
 
    Input:
        scene_description: text description of what the camera sees
        max_new_tokens: max tokens to generate (default 400)
 
    Output:
        JSON incident report with severity, category, confidence etc.
    """
    if not request.scene_description.strip():
        raise HTTPException(status_code=400, detail="scene_description cannot be empty")
 
    if len(request.scene_description) > 2000:
        raise HTTPException(status_code=400, detail="scene_description too long (max 2000 chars)")
 
    start_time = time.time()
 
    result = run_inference(
        request.scene_description,
        request.max_new_tokens
    )
 
    inference_time = round(time.time() - start_time, 2)
 
    if "error" in result:
        raise HTTPException(status_code=500, detail=result)
 
    # Add metadata
    result["_inference_time_seconds"] = inference_time
    result["_model"] = HF_MODEL_ID
 
    return result
 
 

from PIL import Image
import base64 as _base64
from io import BytesIO


class VisionAnalyzeRequest(BaseModel):
    image_base64: str
    mime_type: Optional[str] = "image/jpeg"
    max_new_tokens: Optional[int] = 400


SCENE_DESCRIPTION_PROMPT = (
    "You are a surveillance camera assistant. Describe this camera frame in "
    "precise detail. Cover: who is present (count, clothing, actions), what is "
    "happening, location context (indoor/outdoor, lighting, setting), any objects "
    "of interest. Write 2-4 clear sentences. Plain text only — no JSON, no lists."
)


@app.post("/analyze_vision")
def analyze_vision(request: VisionAnalyzeRequest):
    """
    Analyze a surveillance frame image directly — no AI Studio API key required.

    Step 1: Gemma 4 vision describes the scene in plain text.
    Step 2: Finetuned model converts description to structured incident JSON.

    Input:  { "image_base64": "<base64 JPEG/PNG>", "mime_type": "image/jpeg" }
    Output: Same JSON schema as POST /analyze.
    """
    try:
        image_bytes = _base64.b64decode(request.image_base64)
        image = Image.open(BytesIO(image_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image: {e}")

    # Step 1 — vision: image → plain text scene description
    vision_messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": SCENE_DESCRIPTION_PROMPT},
            ],
        }
    ]
    inputs = tokenizer.apply_chat_template(
        vision_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        desc_out = model.generate(
            **inputs,
            max_new_tokens=200,
            use_cache=True,
            temperature=1.0,
        )
    scene_description = tokenizer.decode(
        desc_out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # Step 2 — text: scene description → structured incident JSON (finetuned model)
    start = time.time()
    result = run_inference(scene_description, request.max_new_tokens)
    if "error" in result:
        raise HTTPException(status_code=500, detail=result)

    result["_inference_time_seconds"] = round(time.time() - start, 2)
    result["_model"] = HF_MODEL_ID
    return result


print("FastAPI app defined")
print("Endpoints:")
print("  GET  /          → info")
print("  GET  /health    → health check")
print("  POST /analyze        → run inference (text scene description)")
print("  POST /analyze_vision → run inference (base64 image input)")
print("  GET  /docs      → swagger UI")
 

FastAPI app defined
Endpoints:
  GET  /          → info
  GET  /health    → health check
  POST /analyze        → run inference (text scene description)
  POST /analyze_vision → run inference (base64 image input)
  GET  /docs      → swagger UI


In [ ]:
# -----------------------------------------------------------------------------
# CELL 5 — Start server with ngrok
# -----------------------------------------------------------------------------
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf
import threading
 
NGROK_TOKEN = ""  # ← get free token at ngrok.com
 
# Configure ngrok
conf.get_default().auth_token = NGROK_TOKEN
nest_asyncio.apply()
 
# Kill any existing ngrok tunnels
ngrok.kill()
 
# Start FastAPI in background thread
PORT = 8000
 
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="error")
 
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
 
import time
time.sleep(3)  # wait for server to start
 
# Open ngrok tunnel
tunnel = ngrok.connect(PORT, "http")
PUBLIC_URL = tunnel.public_url
 
print("=" * 60)
print("SERVER IS RUNNING")
print("=" * 60)
print(f"\nPublic URL: {PUBLIC_URL}")
print(f"\nUpdate your local .env file:")
print(f"  GEMMA_ENDPOINT={PUBLIC_URL}")
print(f"\nTest it right now:")
print(f'  curl {PUBLIC_URL}/health')
print(f'  curl -X POST {PUBLIC_URL}/analyze \\')
print(f'    -H "Content-Type: application/json" \\')
print(f'    -d \'{{"scene_description": "Two men fighting near entrance"}}\'')
print(f"\nSwagger UI: {PUBLIC_URL}/docs")
print("=" * 60)
print("\nKEEP THIS NOTEBOOK RUNNING DURING DEMO")
print("If it stops — rerun this cell to get a new URL")
 

SERVER IS RUNNING

Public URL: https://paddle-probing-march.ngrok-free.dev

Update your local .env file:
  GEMMA_ENDPOINT=https://paddle-probing-march.ngrok-free.dev

Test it right now:
  curl https://paddle-probing-march.ngrok-free.dev/health
  curl -X POST https://paddle-probing-march.ngrok-free.dev/analyze \
    -H "Content-Type: application/json" \
    -d '{"scene_description": "Two men fighting near entrance"}'

Swagger UI: https://paddle-probing-march.ngrok-free.dev/docs

KEEP THIS NOTEBOOK RUNNING DURING DEMO
If it stops — rerun this cell to get a new URL


In [6]:
# -----------------------------------------------------------------------------
# CELL 6 — Test the live endpoint
# -----------------------------------------------------------------------------
import requests
 
print(f"Testing live endpoint at {PUBLIC_URL}...\n")
 
# Test 1 — health check
r = requests.get(f"{PUBLIC_URL}/health")
print(f"Health check: {r.status_code}")
print(f"  {r.json()}\n")
 
# Test 2 — analyze a crime scene
test_payload = {
    "scene_description": "A man in a dark hoodie is breaking a car window with a rock in the parking lot. He is looking around nervously."
}
 
r = requests.post(f"{PUBLIC_URL}/analyze", json=test_payload)
result = r.json()
 
print(f"Crime scene test: {r.status_code}")
print(f"  Severity:  {result.get('severity')}")
print(f"  Category:  {result.get('category')}")
print(f"  Confidence:{result.get('confidence')}")
print(f"  Action:    {result.get('recommended_action')}")
print(f"  Time:      {result.get('_inference_time_seconds')}s\n")
 
# Test 3 — normal scene
test_payload_2 = {
    "scene_description": "An empty hallway with fluorescent lighting. No people visible. All doors are closed."
}
 
r2 = requests.post(f"{PUBLIC_URL}/analyze", json=test_payload_2)
result2 = r2.json()
 
print(f"Normal scene test: {r2.status_code}")
print(f"  Severity:  {result2.get('severity')}")
print(f"  Category:  {result2.get('category')}")
print(f"  Action:    {result2.get('recommended_action')}")
 
print("\nAll tests passed — endpoint is live and working!")
print(f"\nFinal URL for your .env: {PUBLIC_URL}")
 

Testing live endpoint at https://paddle-probing-march.ngrok-free.dev...

Health check: 200
  {'status': 'healthy', 'model': 'Dharya/secure_sight-gemma4-security', 'gpu': 'Tesla T4', 'timestamp': 1777549840.4812505}

Crime scene test: 200
  Severity:  CRITICAL
  Category:  Crime
  Confidence:98
  Action:    Dispatch immediate security patrol to the location, initiate silent audio recording, and prepare for physical intervention if the attack continues.
  Time:      33.0s

Normal scene test: 200
  Severity:  CLEAR
  Category:  Normal
  Action:    Continue routine patrol; no immediate action required.

All tests passed — endpoint is live and working!

Final URL for your .env: https://paddle-probing-march.ngrok-free.dev


In [9]:
# -----------------------------------------------------------------------------
# CELL 7 — Keep alive (run this if session is idle for too long)
# -----------------------------------------------------------------------------
# Kaggle sessions time out after ~9 hours of inactivity
# Run this cell periodically to keep the session alive
# Or just rerun Cell 5 if the tunnel dies — you get a new URL
 
import time
 
print("Keep-alive cell — run this every few hours if needed")
print("Or just rerun Cell 5 if ngrok tunnel dies\n")
 
# Ping the health endpoint every 30 minutes to keep session alive
# Uncomment the loop below if you want automatic keep-alive
# while True:
#     r = requests.get(f"{PUBLIC_URL}/health")
#     print(f"{time.strftime('%H:%M:%S')} — health: {r.status_code}")
#     time.sleep(1800)  # 30 minutes
 
print("To keep alive manually:")
print(f"  curl {PUBLIC_URL}/health")

Keep-alive cell — run this every few hours if needed
Or just rerun Cell 5 if ngrok tunnel dies

To keep alive manually:
  curl https://paddle-probing-march.ngrok-free.dev/health
